## WEEK 6  – ASSIGNMENT

### Objective:
Understand Spark architecture and perform efficient data processing using transformations, filtering, schema handling, and optimized file formats. 

##### Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application. 

In a Spark application, the Driver, Cluster Manager, and Executors work together to process data efficiently.

- Driver: The Driver is the main program that controls the Spark application.It creates the SparkSession,converts the user's code into a series of tasks, schedules those tasks,and collects the results from the executors.
- Cluster Manager: The Cluster Manager is responsible for managing the resources of the cluster.It allocates CPU cores and memory to Spark applications and launches executors on the worker nodes.
- Executor: Executors are worker processes that run on the worker nodes.They execute the tasks assigned by the Driver, perform computations,store intermediate data in memory or disk,and send the results back to the Driver.

##### Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

Spark uses Lazy Evaluation, which means that transformations are not executed immediately. Instead,Spark records all the transformations in a Directed Acyclic Graph  and waits until an action (such as show(), collect(), or count()) is called.

When an action is triggered, Spark analyzes the entire DAG and optimizes the execution plan by combining multiple transformations, eliminating unnecessary operations, and reducing data movement. This minimizes disk I/O,network communication, and memory usage, resulting in faster processing of large datasets.

##### Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled. 

In [0]:
df = spark.read.csv("/Volumes/workspace/default/source/dataset.csv",header=True,inferSchema=True)

In [0]:
df.display()

product_id,price,category,status,amount,region,priority,user_id,old_name,base_price,order_date
P1001,11.12,Books,Pending,33.36,South,High,U98696,Rachel Johnson,11.12,2025-10-14
P1002,108.1,Electronics,Pending,216.2,North,Low,U95181,Rachel Lewis,108.1,2025-04-15
P1003,45.34,Toys,Pending,45.34,West,Medium,U38221,Kevin Johnson,45.34,2025-02-13
P1004,750.01,Electronics,Cancelled,2250.03,North,Low,U26361,Mike Johnson,750.01,2025-09-10
P1005,318.03,Sports,Pending,954.09,North,High,U47930,Charlie Taylor,318.03,2025-02-13
P1006,46.43,Groceries,Pending,139.29,East,Medium,U44993,Charlie King,46.43,2025-11-06
P1007,110.72,Toys,Refunded,221.44,West,Medium,U93886,Rachel Taylor,110.72,2025-11-11
P1008,386.47,Sports,Pending,386.47,North,Medium,U18675,Grace King,386.47,2025-12-11
P1009,134.53,Clothing,Refunded,538.12,South,Medium,U83579,Rachel Anderson,134.53,2025-12-19
P1010,2703.69,Furniture,Cancelled,10814.76,South,High,U21915,Bob Johnson,2703.69,2025-03-21


In [0]:
df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- category: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- order_date: date (nullable = true)



##### Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance? 

CSV is a row-based storage format where data is stored row by row in plain text. It is easy to read but has larger file sizes and slower query performance.

Parquet is a columnar storage format where data is stored column by column in a compressed binary format.It supports efficient compression and encoding.

Why it matters for performance: Since Parquet stores data by columns,Spark reads only the required columns instead of the entire dataset.This reduces disk I/O, memory usage,and network transfer,making queries much faster than CSV,especially for large datasets.

##### 5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

In [0]:
from pyspark.sql.functions import col

df.select("product_id","price").filter(col("category")=="Electronics").display()

product_id,price
P1002,108.1
P1004,750.01
P1012,1376.55
P1017,533.42
P1031,700.44
P1038,1213.31
P1039,1432.13
P1048,1441.41
P1053,1303.36
P1055,1573.4


##### Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double. 

In [0]:
from pyspark.sql.functions import col
df.withColumnRenamed("old_name", "new_name") \
    .withColumn("price", col("price").cast("double")).limit(20).display()


product_id,price,category,status,amount,region,priority,user_id,new_name,base_price,order_date
P1001,11.12,Books,Pending,33.36,South,High,U98696,Rachel Johnson,11.12,2025-10-14
P1002,108.1,Electronics,Pending,216.2,North,Low,U95181,Rachel Lewis,108.1,2025-04-15
P1003,45.34,Toys,Pending,45.34,West,Medium,U38221,Kevin Johnson,45.34,2025-02-13
P1004,750.01,Electronics,Cancelled,2250.03,North,Low,U26361,Mike Johnson,750.01,2025-09-10
P1005,318.03,Sports,Pending,954.09,North,High,U47930,Charlie Taylor,318.03,2025-02-13
P1006,46.43,Groceries,Pending,139.29,East,Medium,U44993,Charlie King,46.43,2025-11-06
P1007,110.72,Toys,Refunded,221.44,West,Medium,U93886,Rachel Taylor,110.72,2025-11-11
P1008,386.47,Sports,Pending,386.47,North,Medium,U18675,Grace King,386.47,2025-12-11
P1009,134.53,Clothing,Refunded,538.12,South,Medium,U83579,Rachel Anderson,134.53,2025-12-19
P1010,2703.69,Furniture,Cancelled,10814.76,South,High,U21915,Bob Johnson,2703.69,2025-03-21


##### Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails? 

Spark provides fault tolerance using a Lineage Graph,also known as a Directed Acyclic Graph.The DAG records all the transformations performed on the data instead of storing multiple copies of intermediate results.

If a worker node fails and some data partitions are lost,Spark uses the Lineage Graph to identify how those partitions were created.It then recomputes only the lost partitions from the original data by re-executing the required transformations, rather than recomputing the entire dataset.

##### Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000. 

In [0]:

from pyspark.sql.functions import col
df.filter((col("status")=="Completed") &(col("amount")>1000)).display()

product_id,price,category,status,amount,region,priority,user_id,old_name,base_price,order_date
P1050,1229.48,Furniture,Completed,6147.4,West,Medium,null,Laura Anderson,1229.48,2025-07-28
P1055,1573.4,Electronics,Completed,1573.4,South,High,U91439,Eva Taylor,1573.4,2025-03-16
P1075,1369.5,Furniture,Completed,6847.5,West,Low,U42816,Alice Johnson,1369.5,2025-04-22
P1076,431.65,Sports,Completed,2158.25,East,Low,null,Frank Walker,431.65,2025-09-21
P1087,1754.67,Electronics,Completed,7018.68,West,High,U63736,Eva Johnson,1754.67,2025-08-26
P1109,1561.13,Furniture,Completed,7805.65,East,Medium,U14082,Bob Walker,1561.13,2025-07-14
P1115,336.02,Sports,Completed,1344.08,East,Low,U23579,Steve Young,336.02,2025-04-05
P1117,314.64,Sports,Completed,1573.2,East,High,U50499,Kevin Clark,314.64,2025-01-06
P1142,2634.94,Furniture,Completed,13174.7,North,Medium,U31012,Ivan Walker,2634.94,2025-09-05
P1162,931.48,Electronics,Completed,3725.92,East,Low,U18511,Rachel Young,931.48,2025-05-10


##### Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Predicate Pushdown is an optimization technique used by Spark when reading Parquet files. Instead of reading the entire dataset into memory and then applying filters, Spark pushes the filter conditions (predicates) down to the Parquet reader.
As a result, only the rows or row groups that satisfy the filter condition are read from disk, while the remaining data is skipped.

Effect on memory and performance:
- Reduces the amount of data loaded into memory.
- Minimizes disk I/O by reading only the required data.
- Speeds up query execution.
- Improves overall performance, especially when working with large datasets.

##### Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [0]:
from pyspark.sql.functions import col
df = df.withColumn("final_price", col("base_price") * 1.18)
df.display()

product_id,price,category,status,amount,region,priority,user_id,old_name,base_price,order_date,final_price
P1001,11.12,Books,Pending,33.36,South,High,U98696,Rachel Johnson,11.12,2025-10-14,13.121599999999999
P1002,108.1,Electronics,Pending,216.2,North,Low,U95181,Rachel Lewis,108.1,2025-04-15,127.55799999999999
P1003,45.34,Toys,Pending,45.34,West,Medium,U38221,Kevin Johnson,45.34,2025-02-13,53.501200000000004
P1004,750.01,Electronics,Cancelled,2250.03,North,Low,U26361,Mike Johnson,750.01,2025-09-10,885.0118
P1005,318.03,Sports,Pending,954.09,North,High,U47930,Charlie Taylor,318.03,2025-02-13,375.27539999999993
P1006,46.43,Groceries,Pending,139.29,East,Medium,U44993,Charlie King,46.43,2025-11-06,54.7874
P1007,110.72,Toys,Refunded,221.44,West,Medium,U93886,Rachel Taylor,110.72,2025-11-11,130.6496
P1008,386.47,Sports,Pending,386.47,North,Medium,U18675,Grace King,386.47,2025-12-11,456.0346
P1009,134.53,Clothing,Refunded,538.12,South,Medium,U83579,Rachel Anderson,134.53,2025-12-19,158.7454
P1010,2703.69,Furniture,Cancelled,10814.76,South,High,U21915,Bob Johnson,2703.69,2025-03-21,3190.3541999999998


##### Q11: What is the difference between Transformations and Actions? Provide two examples of each.

Transformations are operations that create a new DataFrame or RDD from an existing one without executing the computation immediately.Spark records these operations and performs them only when an action is called.This behavior is known as lazy evaluation.

Examples of Transformations:
filter(),select()

Actions are operations that trigger the execution of all pending transformations and produce a result or write data.When an action is called.

Examples of Actions:
show(),collect()

##### Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [0]:
df.write.mode("overwrite").parquet("/Volumes/workspace/default/source/parquet_data")

In [0]:
from pyspark.sql.functions import col
spark.read.parquet("/Volumes/workspace/default/source/parquet_data")\
.filter(col("user_id").isNotNull())\
.write.mode("overwrite")\
.option("header", "true")\
.csv("/Volumes/workspace/default/source/output_csv")

##### Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode? 

Client Mode:
In Client Mode,the Driver runs on the client machine that submits the Spark application. The client communicates with the cluster, schedules tasks,and collects the results. The client machine must remain connected while the application is running.This mode is mainly used for development,testing,and debugging.

Cluster Mode:
In Cluster Mode,the Driver runs inside the cluster on one of the worker nodes.The cluster manages the entire application,so the client can disconnect after submitting the job.This mode is more reliable and is commonly used for production environments.

##### Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [0]:
from pyspark.sql.functions import col
df.filter((col("region")=="North") |(col("priority")=="High")).display()

product_id,price,category,status,amount,region,priority,user_id,old_name,base_price,order_date,final_price
P1001,11.12,Books,Pending,33.36,South,High,U98696,Rachel Johnson,11.12,2025-10-14,13.121599999999999
P1002,108.1,Electronics,Pending,216.2,North,Low,U95181,Rachel Lewis,108.1,2025-04-15,127.55799999999999
P1004,750.01,Electronics,Cancelled,2250.03,North,Low,U26361,Mike Johnson,750.01,2025-09-10,885.0118
P1005,318.03,Sports,Pending,954.09,North,High,U47930,Charlie Taylor,318.03,2025-02-13,375.27539999999993
P1008,386.47,Sports,Pending,386.47,North,Medium,U18675,Grace King,386.47,2025-12-11,456.0346
P1010,2703.69,Furniture,Cancelled,10814.76,South,High,U21915,Bob Johnson,2703.69,2025-03-21,3190.3541999999998
P1012,1376.55,Electronics,Cancelled,1376.55,East,High,U30730,Oscar Smith,1376.55,2025-12-24,1624.329
P1014,461.99,Sports,Cancelled,461.99,West,High,U57576,Julia Taylor,461.99,2025-01-08,545.1482
P1015,142.31,Toys,Refunded,142.31,North,Low,U26828,Paula Young,142.31,2025-03-09,167.92579999999998
P1017,533.42,Electronics,Cancelled,533.42,North,Low,U87128,Hannah Smith,533.42,2025-02-23,629.4355999999999


##### Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset? 

.show(5) is safer because it displays only the first five rows of the dataset.It retrieves a small amount of data,making it fast and memory efficient.

In contrast,.collect() retrieves all the rows from the dataset and brings them to the Driver's memory.On a multi-terabyte dataset, this can consume a large amount of memory, slow down the application, or even cause the Driver to crash with an OutOfMemoryError.